# 04 · Evaluation

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| 3. Model training | `03_model_training.ipynb` |
| **4. Evaluation** | **this notebook** |

**Inputs** (written by `python scripts/run_phase1.py`, the single source of truth):
* `results/predictions.csv.gz`: raw scores of every model on every flight, plus each outcome's availability date. Calibration (including rolling recalibration), metrics, policies, drift and the bootstrap are **recomputed here** with the same `aerocast.pipeline` functions and checked against the saved tables.
* Ablation, permutation importance, gain/coefficients, target sensitivity, PSI and adversarial validation need refitted models, so they are **loaded** from `results/`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import io, json
import numpy as np, pandas as pd
from IPython.display import display
from aerocast import config, pipeline, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 100)
R = config.RESULTS_DIR
S = json.loads((R / "summary.json").read_text())
FINAL = config.FINAL_MODEL

def same_as_saved(df, name):
    """Exact comparison with a saved CSV (round-trip float parsing)."""
    a = pd.read_csv(io.StringIO(df.to_csv(index=False)), float_precision="round_trip")
    b = pd.read_csv(R / name, float_precision="round_trip")
    pd.testing.assert_frame_equal(a, b)
    return f"OK: identical to results/{name}"

labels, raw_preds = pipeline.load_predictions()
print("final model (fixed before test):", S["final_model"])

final model (fixed before test): {'model': 'logreg', 'rule': 'higher mean rolling-CV PR-AUC', 'fixed_before_test': True}


## 0. Development protocol (disclosure)

The test window (Nov–Dec) is meant to be touched once. Here is an honest list of what was seen during development, and how each decision now rests on **CV / Sep–Oct evidence only**.

**Diagnostics that printed test metrics before the design was frozen:**
1. **First run** (single-September tuning design): test PR-AUC and calibration of all models were printed. The design was abandoned because of the September evidence (XGBoost stopped at 13 trees), but test numbers had been seen.
2. **Month features:** a with/without comparison printed test PR-AUC (0.377 vs 0.382), and that was the original justification for removing them. **Withdrawn.** Re-justified by rolling CV (notebook 02, §7).
3. **Feature-subset variants** (dropping log-counts, IDs, day-level history, congestion; a minimal set) printed calibration-window *and* test PR-AUC and test recall@precision. These shaped the "day-level features memorise days" hypothesis. **Re-justified** by the distinct-value analysis and the CV ablation (notebook 02, §8; §8 below).
4. **Precision target:** 0.50 → 0.40 was decided because 0.50 is unreachable on Sep–Oct (base rate 15.7%), but the same diagnostic also printed test recall at 0.35/0.40/0.50.
5. **Calibration:** test ECE of raw vs Platt vs isotonic was seen *before* any selection rule existed. The "fit Sep → check Oct" rule, the shift-aware Platt variant (evaluated on test ECE), and the 0.005-margin rule (this review) were all introduced **after** that.
6. **This review round:** the full pipeline (which also computes test metrics) ran once while the final-model comparison still used XGBoost's optimistic in-fold early-stopping score. The comparison was corrected for a methodological reason identified from CV alone (notebook 03, §7). That correction reversed the choice to LogReg.

**Decided without test evidence:** the decision clock (and clock vs 2-day lag, CV), smoothing m and window (CV), C and XGBoost configuration (CV), final model (CV as deployed), calibration margin rule and its outcome (Sep→Oct), rolling-recalibration window N = 28 (a priori), daily top-10% policy (a priori capacity assumption), and the precision target (Sep–Oct).

**Implication.** Test numbers are not perfectly clean confirmations. The design was iterated with test results visible on several occasions, so treat small test differences with caution. Every conclusion below is stated with its CV or Sep–Oct basis.

## 1. Metric theory in brief

* **PR-AUC (average precision).** sklearn's `average_precision_score` computes $AP = \sum_k (R_k - R_{k-1})\,P_k$, a step-wise sum over thresholds. It is *not* the trapezoidal area under the PR curve, which interpolates linearly between points and is optimistic because PR curves aren't convex. "PR-AUC" in the tables means AP. Its no-skill baseline is the **base rate**.
* **PR vs ROC under imbalance.** ROC-AUC is the probability that a random late flight is ranked above a random on-time flight. It ignores the base rate, so the same ROC can mean very different precision when positives are rare. PR curves put precision on the axis, which is what an operations desk experiences. Both are reported.
* **ECE** (Guo et al., 2017): $\mathrm{ECE} = \sum_b \frac{|B_b|}{n}\,\big|\bar y_b - \bar p_b\big|$ over bins of predicted probability.
  * It's a **biased estimator**: sampling noise in each bin's $\bar y_b$ always adds, so ECE > 0 even for a perfectly calibrated model in finite samples, and the bias grows with the number of bins.
  * Results also depend on the binning. **Equal-width** bins (15) leave the sparse high-risk tail with few flights; **equal-mass** bins (quantiles, `ece_equal_mass`) give every bin the same count. Both are reported, and the Brier score (proper, binning-free) is reported alongside.
* **Platt / temperature scaling vs isotonic.** Platt fits $\sigma(a\,\mathrm{logit}(p)+b)$: 2 parameters, monotone, smooth, data-efficient. Temperature scaling is its 1-parameter special case ($b=0$), the multi-class tool in Guo et al. Isotonic regression fits any monotone step function: more flexible, but needs more data and can overfit. All of them assume the calibration data are **distributed like deployment data**.
* **Why the bootstrap resamples whole days.** Flights on the same day share weather, air-traffic programmes and cascading delays, so their errors are correlated. Resampling individual flights treats about 900 correlated flights as independent and gives intervals that are far too narrow. Resampling **days** keeps each day's dependence intact (a block bootstrap). Model differences are computed on the *same* resampled days (paired).

## 2. Calibration: a static rule fixed in advance, plus causal rolling recalibration

**Static rule (fixed before test, `config.CALIB_MIN_ECE_GAIN = 0.005`):** fit Platt and isotonic on **September**, measure ECE on **October**, and keep the **raw** scores unless the better recalibrator lowers October ECE by more than 0.005. The winner is then refit on all of Sep–Oct. This replaces the earlier "lowest October ECE wins" rule, which picked Platt for LogReg on a 0.0007 near-tie.

**Rolling recalibration (alternative, evaluated on test):** for every day D, refit Platt on outcomes known by D's decision clock over the trailing N days, and apply it to D's flights. It's causal, using the same availability dates as the features. **N = 28 was fixed a priori**; 14 and 56 are reported as sensitivity only.

In [2]:
final_preds, choice, calib_tbl = pipeline.calibrate(labels, raw_preds)
print(same_as_saved(calib_tbl, "calibrator_selection.csv"))
print("static-rule choice:", choice)
calib_tbl.set_index(["model", "calibrator"]).round(4)

OK: identical to results/calibrator_selection.csv
static-rule choice: {'logreg': 'platt', 'xgboost': 'raw'}


check_ece  check_ece_equal_mass  check_brier  gain_vs_raw
model   calibrator                                                           
logreg  raw            0.0609                0.0614       0.1404       0.0000
        platt          0.0349                0.0349       0.1378       0.0260
        isotonic       0.0362                0.0356       0.1386       0.0247
xgboost raw            0.0220                0.0245       0.1386       0.0000
        platt          0.0465                0.0465       0.1411      -0.0245
        isotonic       0.0468                0.0468       0.1413      -0.0248

* **LogReg → Platt, by the rule.** Raw LogReg over-predicts in October (it carries the 26.5% training base rate into a 17.6% month). Platt improves October ECE by **0.026**, well above the 0.005 margin, so the rule clearly adopts it.
* **XGBoost → raw.** Recalibration makes October worse.

## 3. Test metrics (Nov–Dec)

In [3]:
thresholds = pipeline.frozen_thresholds(labels, final_preds)
results = pipeline.metrics_table(labels, final_preds, thresholds)
print(same_as_saved(results, "metrics_all.csv"))
pipeline.headline_table(results)[pipeline.SHOW_COLS]

OK: identical to results/metrics_all.csv


,pr_auc,roc_auc,brier,ece,ece_equal_mass,mean_pred,base_rate,oracle_recall@p0.30,oracle_recall@p0.40,oracle_recall@p0.50
model,,,,,,,,,,
prior,0.2583,0.5000,0.1916,0.0066,0.0842,0.2649,0.2583,0.0000,0.0000,0.0000
route_history,0.3534,0.6208,0.1869,0.0373,0.0380,0.2420,0.2583,0.7868,0.2444,0.0163
logreg_cal,0.3937,0.6537,0.1872,0.0815,0.0815,0.1768,0.2583,0.8469,0.4303,0.1294
xgboost_cal,0.4070,0.6619,0.1797,0.0267,0.0266,0.2317,0.2583,0.8633,0.4740,0.1613


`*_cal` = the static-rule variant. **`oracle_recall@p`** is the best recall at precision ≥ p, with the threshold **tuned on the test labels themselves**. It's a point on the PR curve, valid **only for comparing models**, and not an achievable operating result (that's §7).

* The final model (**LogReg**) has test PR-AUC **0.394**. XGBoost scores 0.407, the route-history heuristic 0.353, and the prior 0.258.
* The rule-chosen Platt calibration gives LogReg a poor test ECE (0.082). §5 explains why.

## 4. LogReg vs XGBoost: day-block bootstrap

In [4]:
boot = pipeline.bootstrap_compare(labels, final_preds)
assert json.loads(json.dumps(boot)) == S["bootstrap_test"], "bootstrap differs from summary.json"
pd.DataFrame({k: {"XGBoost 95% CI": np.round(v["xgboost_ci95"], 3), "LogReg 95% CI": np.round(v["logreg_ci95"], 3),
                  "XGB − LR 95% CI": np.round(v["diff_xgboost_minus_logreg_ci95"], 3),
                  "P(XGB not better)": v["p_xgboost_not_better"]} for k, v in boot.items()}).T

,XGBoost 95% CI,LogReg 95% CI,XGB − LR 95% CI,P(XGB not better)
pr_auc,"[0.348, 0.465]","[0.34, 0.45]","[-0.01, 0.041]",0.156
roc_auc,"[0.633, 0.691]","[0.62, 0.682]","[-0.012, 0.029]",0.208


On test, XGBoost is ahead by about 0.013 PR-AUC, but the **95% CI includes 0** (about −0.010 to +0.041), so the two are statistically indistinguishable. Together with the CV result (LogReg ahead as deployed), the honest conclusion is the same as before: **hand-built day-ahead summaries leave little interaction structure for trees to exploit**. That's the gap the deep-learning phase targets.

## 5. PR/ROC curves, reliability, and the calibration study

In [5]:
figs = pipeline.evaluation_figures(labels, final_preds,
                                   pipeline.monthly_drift(labels, final_preds),
                                   pd.read_csv(R / "xgb_gain_importance.csv", index_col=0)["gain"],
                                   pd.read_csv(R / "permutation_importance_groups.csv"),
                                   pd.read_csv(R / "ablation.csv"), pd.read_csv(R / "psi.csv"))
for name, fig in figs.items():
    plots.save(fig, name)
figs["pr_roc_test"]

<Figure size 850x360 with 2 Axes>

In [6]:
figs["reliability_logreg_test"]

<Figure size 440x480 with 2 Axes>

In [7]:
figs["reliability_xgboost_test"]

<Figure size 440x480 with 2 Axes>

In [8]:
study = pipeline.calibration_study(results)
study.round(4)

brier                       ece                   ece_equal_mass                   mean_pred                  
split                test test_m11 test_m12    test test_m11 test_m12           test test_m11 test_m12      test test_m11 test_m12
model                                                                                                                             
logreg_raw         0.1811   0.1454   0.2168  0.0190   0.0553   0.0595         0.0216   0.0569   0.0601    0.2581   0.2349   0.2812
logreg_platt       0.1872   0.1418   0.2326  0.0815   0.0208   0.1421         0.0815   0.0208   0.1421    0.1768   0.1602   0.1933
logreg_isotonic    0.1873   0.1419   0.2327  0.0817   0.0226   0.1416         0.0802   0.0209   0.1394    0.1781   0.1601   0.1961
logreg_rolling14   0.1807   0.1431   0.2181  0.0215   0.0110   0.0385         0.0230   0.0116   0.0411    0.2512   0.1752   0.3270
logreg_rolling28   0.1809   0.1419   0.2199  0.0269   0.0065   0.0499         0.0265   0.0101   0.0496    0.2318   0.1772   0.2863
logreg_rolling56   0.1820   0.1416   0.2223  0.0510   0.0135   0.0886         0.0510   0.0133   0.0886    0.2073   0.1677   0.2469
xgboost_raw        0.1797   0.1435   0.2158  0.0267   0.0191   0.0642         0.0266   0.0207   0.0642    0.2317   0.1922   0.2712
xgboost_platt      0.1864   0.1440   0.2286  0.0812   0.0329   0.1309         0.0812   0.0314   0.1309    0.1771   0.1496   0.2046
xgboost_isotonic   0.1859   0.1443   0.2274  0.0787   0.0341   0.1258         0.0787   0.0316   0.1258    0.1796   0.1494   0.2097
xgboost_rolling14  0.1802   0.1440   0.2163  0.0227   0.0125   0.0429         0.0231   0.0155   0.0421    0.2588   0.1734   0.3441
xgboost_rolling28  0.1803   0.1438   0.2168  0.0241   0.0130   0.0382         0.0231   0.0143   0.0384    0.2444   0.1761   0.3127
xgboost_rolling56  0.1803   0.1433   0.2172  0.0397   0.0220   0.0580         0.0397   0.0238   0.0580    0.2186   0.1596   0.2775

**Every calibration variant on test (Nov–Dec), with Nov and Dec separately:**
* **Static recalibration fails under the shift.** Platt and isotonic were fitted on calm Sep–Oct (15.7% late), so they scale risk *down*. That's correct for November (LogReg Platt ECE 0.021) and badly wrong for December (0.142, base rate 33.5%). The rule was applied as fixed, and its test result is reported as is.
* **Raw scores** happen to be well calibrated on Nov–Dec overall (LogReg ECE 0.019), because the training base rate (26.5%) is close to the test base rate (25.8%). That's a coincidence of this split, not a property you could count on in deployment.
* **Causal rolling Platt** (N = 28, fixed a priori) **adapts**: LogReg test ECE 0.027 (Nov 0.007, Dec 0.050), XGBoost 0.024. It tracks the base rate as it changes. Shorter windows adapt faster (N = 14: 0.022) and longer ones lag (N = 56: 0.051). Those two were not selected; they're reported only.
* **Lesson:** post-hoc calibration (Guo et al.) assumes the calibration data look like deployment data. Under temporal prior shift (Ovadia et al., 2019) a frozen calibrator breaks, and the remedy is to **recalibrate continuously on recent known outcomes**.

## 6. Monthly calibration drift (final model, raw scores)

In [9]:
drift = pipeline.monthly_drift(labels, final_preds)
print(same_as_saved(drift, "monthly_calibration_drift.csv"))
figs["monthly_drift_final"]

OK: identical to results/monthly_calibration_drift.csv


<Figure size 650x320 with 1 Axes>

In [10]:
drift.pivot_table(index="month", columns="model", values=["observed", "predicted"]).round(3)

observed                                            predicted                                        
model logreg_platt logreg_raw logreg_rolling28 xgboost_raw logreg_platt logreg_raw logreg_rolling28 xgboost_raw
month                                                                                                          
1            0.238      0.238            0.238       0.238        0.142      0.207              NaN       0.236
2            0.238      0.238            0.238       0.238        0.174      0.254              NaN       0.258
3            0.236      0.236            0.236       0.236        0.173      0.253              NaN       0.249
4            0.288      0.288            0.288       0.288        0.179      0.262              NaN       0.268
5            0.227      0.227            0.227       0.227        0.182      0.265              NaN       0.236
6            0.315      0.315            0.315       0.315        0.190      0.277              NaN       0.286
7            0.322      0.322            0.322       0.322        0.215      0.308              NaN       0.319
8            0.246      0.246            0.246       0.246        0.189      0.276              NaN       0.257
9            0.137      0.137            0.137       0.137        0.155      0.227            0.159       0.210
10           0.176      0.176            0.176       0.176        0.159      0.233            0.154       0.196
11           0.181      0.181            0.181       0.181        0.160      0.235            0.177       0.192
12           0.335      0.335            0.335       0.335        0.193      0.281            0.286       0.271

The raw predictions follow the **shape** of the season but **compress it toward the middle**. They over-predict in calm Sep–Oct and under-predict December's 33.5%. The rolling-Platt column shows the correction tracking each month.

## 7. Operating policies (fixed before test)

* **PRIMARY: daily capacity policy.** Each day, flag the **10%** highest-risk flights, as an operations desk with fixed review capacity would. It uses only the ranking, so it's immune to base-rate shift. The 10% share is an **a-priori capacity assumption** and was not tuned.
* **SECONDARY: frozen threshold.** On Sep–Oct, take the lowest threshold that reaches precision ≥ 0.40, and keep it fixed for Nov–Dec. The target was **0.50** first, but that is unreachable on Sep–Oct, so it was lowered to 0.40 (a Sep–Oct decision).

In [11]:
policies = pipeline.operating_policies(labels, final_preds, thresholds)
print(same_as_saved(policies, "operating_policies.csv"))
policies.round(3)

OK: identical to results/operating_policies.csv


,model,split,role,policy,flag_rate,precision,recall,lift,threshold
0,route_history,calib,primary,daily top 10%,0.100,0.233,0.149,1.483,NaN
1,route_history,calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.001,0.417,0.002,2.655,0.469
2,route_history,test,primary,daily top 10%,0.101,0.359,0.140,1.389,NaN
3,route_history,test,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.059,0.425,0.097,1.646,0.469
4,logreg_cal,calib,primary,daily top 10%,0.100,0.303,0.194,1.932,NaN
5,logreg_cal,calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.038,0.400,0.097,2.549,0.331
6,logreg_cal,test,primary,daily top 10%,0.101,0.444,0.173,1.720,NaN
7,logreg_cal,test,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.075,0.493,0.143,1.910,0.331
8,xgboost_cal,calib,primary,daily top 10%,0.100,0.294,0.188,1.874,NaN
9,xgboost_cal,calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.015,0.400,0.037,2.550,0.446


**Primary result (final model, test): the daily top-10% policy flags 10% of flights with 44.4% precision and 17.3% recall, a 1.72× lift over random flagging.** Precision is stable in relative terms between Sep–Oct (30%, lift 1.93) and Nov–Dec (44%, lift 1.72). The absolute precision rises with the base rate.

The **frozen threshold** behaves unpredictably under the shift. For LogReg it flags 3.8% of flights in Sep–Oct but 7.5% in Nov–Dec, because the base rate (and therefore the score distribution) moved. A fixed probability cut-off doesn't carry over between seasons.

## 8. Feature-family ablation, both models

Each cumulative step is refit with the CV-chosen hyperparameters. **Claims rest on rolling CV** (mean gain and how many of the 4 folds improve). The test difference, with a paired day-block bootstrap 95% CI, is a confirmation only.

In [12]:
abl = pd.read_csv(R / "ablation.csv")
display(abl.round(4))
figs["ablation_cv"]

,model,features,n_features,cv_pr_auc_mean,cv_pr_auc_std,test_pr_auc,test_roc_auc
0,logreg,schedule + ids,13,0.4191,0.0643,0.3450,0.6126
1,logreg,+ congestion,20,0.4400,0.0729,0.3522,0.6145
2,logreg,+ history_entity,38,0.4595,0.0727,0.3846,0.6428
3,logreg,+ history_day,45,0.4705,0.0706,0.3937,0.6537
4,logreg,history only,25,0.4478,0.0753,0.3903,0.6566
5,xgboost,schedule + ids,13,0.4391,0.0754,0.3430,0.6193
6,xgboost,+ congestion,20,0.4509,0.0886,0.3454,0.6186
7,xgboost,+ history_entity,38,0.4490,0.0769,0.3973,0.6536
8,xgboost,+ history_day,45,0.4450,0.0729,0.4070,0.6619
9,xgboost,history only,25,0.4231,0.0879,0.3798,0.6462


<Figure size 650x370 with 1 Axes>

In [13]:
pd.read_csv(R / "ablation_steps.csv").round(4)

,model,step,cv_gain_mean,cv_folds_improved,test_diff,test_diff_ci_lo,test_diff_ci_hi
0,logreg,+ congestion,0.0209,4/4,0.0072,-0.0092,0.0275
1,logreg,+ history_entity,0.0195,4/4,0.0324,0.0154,0.0493
2,logreg,+ history_day,0.0110,3/4,0.0091,-0.0091,0.0268
3,xgboost,+ congestion,0.0118,3/4,0.0024,-0.0101,0.0147
4,xgboost,+ history_entity,-0.0019,2/4,0.0520,0.0263,0.0772
5,xgboost,+ history_day,-0.0040,3/4,0.0096,-0.0238,0.0454


**Final model (LogReg):** every family helps in CV.
* **+ congestion: +0.021, 4/4 folds.** Consistent with the within-hour analysis in notebook 01. Test confirms the direction, but the CI includes 0.
* **+ entity history: +0.020, 4/4 folds.** Test **confirms** (+0.032, CI 0.015 to 0.049).
* **+ day-level history: +0.011, 3/4 folds.** Test is positive, but the CI includes 0.

**XGBoost** (fixed 143 trees on every subset): only congestion helps in CV. Entity and day-level history show no CV gain, yet entity history is confirmed on test (+0.052, CI excludes 0). CV and test disagree here, so XGBoost's ablation is reported but no claim is based on it.

History alone (LogReg CV 0.448) is weaker than schedule + history, so the families complement each other.

## 9. Feature importance

### 9.1 Grouped permutation importance (Sep–Oct, not test)

Each family is shuffled **together** with one permutation (5 repeats). Correlated features within a family can't stand in for each other, and the history features *are* strongly correlated (notebook 01, §8).

In [14]:
perm = pd.read_csv(R / "permutation_importance_groups.csv")
display(perm.pivot(index="group", columns="model", values="pr_auc_drop").round(4))
figs["perm_importance_groups"]

model,logreg,xgboost
group,,
categorical,0.0376,0.0147
congestion,0.0464,0.0150
history_day,0.0261,0.0089
history_entity,0.0287,0.0178
schedule,0.0423,0.0537


<Figure size 550x220 with 1 Axes>

### 9.2 XGBoost gain (top 20) and LogReg coefficients

In [15]:
figs["xgb_gain_top20"]

<Figure size 550x640 with 1 Axes>

In [16]:
pd.read_csv(R / "logreg_coefficients.csv", index_col=0).head(15)

,coef
num__hist_logn_carrier_14d,0.6607
num__hist_logn_carrier_7d,-0.6349
cat__dest_BOS,-0.5846
cat__dest_PHL,0.4692
cat__origin_EWR,-0.4503
cat__dest_LAX,-0.4051
cat__dest_SNA,-0.3870
cat__dest_ORD,-0.3384
cat__carrier_DL,-0.3182
cat__dest_PSE,-0.3115


* **Grouped permutation** (the primary measure): for the **final model (LogReg)**, scheduled **congestion** (0.046) and **schedule / time of day** (0.042) matter most, followed by the IDs (0.038) and the two history families (about 0.03 each). For XGBoost, schedule dominates (0.054) and every other family is under 0.02. Every family contributes, which agrees with the LogReg ablation in §8.
* **XGBoost gain:** origin × hour history, flight-number history and `dep_hour` lead, i.e. time of day plus fine-grained entity history.
* **LogReg coefficients are conditional and collinear.** The largest pair is `hist_logn_carrier_14d` (+0.66) and `hist_logn_carrier_7d` (−0.63). They have opposite signs on nearly collinear volumes, so together they encode the *change* in a carrier's recent volume, and neither is interpretable alone. This is why grouped permutation, not coefficients, is the importance measure.

## 10. Target sensitivity: counting cancelled and diverted flights as delayed

In [17]:
pd.read_csv(R / "target_sensitivity.csv").round(4)

,target,model,n_test,base_rate,pr_auc,roc_auc,brier,ece,top10_precision,top10_recall,top10_lift,xgb_minus_lr_pr_auc_ci
0,arr_delay>=15 (main),logreg,53991,0.2583,0.3937,0.6537,0.1811,0.0190,0.4444,0.1730,1.7203,"[-0.010, +0.041]"
1,arr_delay>=15 (main),xgboost,53991,0.2583,0.4070,0.6619,0.1797,0.0267,0.4348,0.1692,1.6832,"[-0.010, +0.041]"
2,arr_delay>=15 OR cancelled/diverted,logreg,55403,0.2772,0.4461,0.6609,0.1864,0.0240,0.4784,0.1735,1.7260,"[-0.013, +0.037]"
3,arr_delay>=15 OR cancelled/diverted,xgboost,55403,0.2772,0.4549,0.6694,0.1848,0.0290,0.4754,0.1724,1.7150,"[-0.013, +0.037]"


Counting cancelled and diverted flights as delayed (a passenger-disruption view) raises the base rate from 25.8% to 27.7%. PR-AUC rises with it, mainly because PR-AUC's baseline is the base rate. **The conclusions don't change:**
* XGBoost and LogReg remain indistinguishable (the CI of the difference includes 0);
* the primary policy gives the same lift (about 1.72);
* ROC-AUC barely moves (+0.007).

The findings don't depend on how cancellations are handled.

## 11. Covariate shift: PSI and adversarial validation

In [18]:
psi = pd.read_csv(R / "psi.csv")
display(psi.head(12).round(3))
figs["psi_top15"]

,feature,group,psi_calib,psi_test
0,days_to_holiday,schedule,0.261,4.769
1,hist_logn_origin_7d,history_entity,2.269,1.265
2,hist_rate_network_7d,history_day,5.147,1.058
3,hist_rate_origin_7d,history_entity,3.433,0.755
4,sched_origin_day,congestion,2.056,0.610
5,hist_rate_carrier_14d,history_entity,2.631,0.383
6,hist_rate_dest_14d,history_entity,2.143,0.292
7,hist_logn_carrier_14d,history_entity,0.472,0.289
8,hist_cancel_network_1d,history_day,1.229,0.222
9,hist_rate_carrier_7d,history_entity,1.699,0.209


<Figure size 550x500 with 1 Axes>

In [19]:
pd.read_csv(R / "adversarial_validation.csv").round(3)

,comparison,features,n_features,adversarial_auc,top_features
0,train vs calib,all features,45,1.000,"hist_rate_network_7d, hist_rate_dest_14d, hist_rate_carrier_14d, hist_logn_origin_7d"
1,train vs calib,categorical,3,0.562,"dest, carrier, origin"
2,train vs calib,schedule,10,0.785,"days_to_holiday, dow_cos, dow_sin, arr_tod_sin"
3,train vs calib,congestion,7,0.983,"sched_origin_day, sched_carrier_origin_day, sched_origin_hour, sched_origin_3h"
4,train vs calib,history_entity,18,1.000,"hist_rate_origin_7d, hist_rate_dest_14d, hist_rate_carrier_14d, hist_logn_origin_7d"
5,train vs calib,history_day,7,1.000,"hist_rate_network_7d, hist_rate_network_1d, hist_logn_origin_1d, hist_cancel_network_1d"
6,train vs test,all features,45,1.000,"days_to_holiday, origin, hist_logn_origin_7d, hist_rate_network_7d"
7,train vs test,categorical,3,0.562,"dest, carrier, origin"
8,train vs test,schedule,10,0.921,"days_to_holiday, is_weekend, dow_cos, dow_sin"
9,train vs test,congestion,7,0.963,"sched_origin_day, sched_carrier_origin_day, sched_origin_3h, tail_legs_day"


* **PSI** (train as reference; above 0.25 counts as a large shift): the history rates shift strongly into Sep–Oct (they follow the season by design). `days_to_holiday` shifts most into Nov–Dec (Thanksgiving, Christmas and New Year all fall in the test window).
* **Adversarial validation:** a classifier separates training rows from calibration or test rows **perfectly using all features**, and also using **history features alone**. Day-level history takes one value per day, so it identifies the period trivially. That shows *why* the day-level group is a memorisation risk (notebook 02, §8).
* The per-family rows show where the shift lives:
  * Only the **IDs** (carrier, origin, destination) are nearly stable (AUC 0.56), so the route and carrier mix barely changes.
  * **Schedule** features separate the periods quite well (0.79 vs calib, 0.92 vs test), through holiday proximity and the weekday mix.
  * **Congestion** features almost perfectly (0.96–0.98), because daily scheduled volumes differ by season. Like day-level history, they take one value per airport-day.
  * **History** separates the periods completely: it encodes the changing conditions by design.
* This is a strong, measured covariate shift, and it's the root cause of the calibration findings in §5.

## 12. Findings

1. **Leak fixed, and it was material.** 31% of D-1 arrivals land after the 20:00 decision time, and they're twice as often late. Removing them costs about 0.018 rolling-CV PR-AUC. A 2-day lag would cost about 0.01 more (CV).
2. **Final model: LogReg** (CV as deployed, 0.4705 vs 0.4450). On test, LogReg PR-AUC is 0.394 vs XGBoost 0.407, statistically indistinguishable. Day-ahead signal is weak and mostly additive.
3. **Every feature family adds CV value for the final model.** Entity history is confirmed on test. Month features don't help (CV).
4. **Calibration:** the static rule fixed in advance adopts Platt for LogReg, which fails in December under the seasonal shift. **Causal rolling recalibration** (N = 28, a priori) keeps test ECE at about 0.025 for both models.
5. **Primary operating policy (daily top 10%): precision 0.444, recall 0.173, lift 1.72** on test. A frozen probability threshold doesn't transfer between seasons.
6. **Robust to the target definition** (cancellations as delays), and the covariate-shift diagnostics explain the calibration behaviour.

## 13. Limitations and assumptions
* **Single year, three origins.** Each season and airport effect is observed once; no out-of-year validation is possible.
* **No weather data.** This is the main missing driver of day-to-day delays.
* **Only NYC departures**, so aircraft rotation is only partly visible.
* **Assumptions:** tail-number assignment known by 20:00 on D-1; US *federal* holiday calendar; cancellations known by the scheduled departure time; destination time zones from a fixed table plus the 2013 DST rule; outcomes known at actual gate arrival.
* **Test was not perfectly clean** (§0). Small test differences should be read with caution.